### 04 — Classical Finance Benchmark: Altman-Style Distress Modeling

#### Objective

Before fully trusting machine learning models, we benchmark classical financial distress theory using Altman-style ratios.

This notebook evaluates:

1. The predictive power of traditional financial distress ratios.
2. Whether classical theory still carries signal.
3. How data-driven linear models compare to fixed historical coefficients.

---

#### Background — Altman Z-Score

The Altman Z-Score (1968) was developed to predict corporate bankruptcy using a linear combination of financial ratios.

Original structure:

Z = w₁X₁ + w₂X₂ + w₃X₃ + w₄X₄ + w₅X₅

Where ratios typically represent:

- Working capital / total assets
- Retained earnings / total assets
- EBIT / total assets
- Equity / liabilities
- Sales / total assets

Altman's model was designed for publicly traded manufacturing firms.

---

#### Why Re-Evaluate Altman?

- The dataset is modern and cross-sectional.
- Financial environments have evolved.
- Firms are heterogeneous across industries.
- Machine learning allows flexible re-weighting.

We test:

1️⃣ Classical fixed-weight approach  
2️⃣ Data-optimized logistic re-estimation using same ratios  

This provides a bridge between financial theory and statistical learning.

In [2]:
import pandas as pd
import numpy as np

from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

In [3]:
data, meta = arff.loadarff("../data/raw/1year.arff")
df = pd.DataFrame(data)

df["class"] = df["class"].apply(lambda x: int(x.decode("utf-8")))

X = df.drop(columns=["class"])
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

### Classical Altman Z-Score Benchmark

The original Altman Z-score (1968) is a linear combination of five financial ratios:

- Working Capital / Total Assets (Attr3)
- Retained Earnings / Total Assets (Attr6)
- EBIT / Total Assets (Attr7)
- Equity / Total Liabilities (Attr8, book value proxy)
- Sales / Total Assets (Attr9)

We evaluate its discriminatory power on the 1-year bankruptcy dataset.

In [4]:
altman_cols = ["Attr3", "Attr6", "Attr7", "Attr8", "Attr9"]

imputer = SimpleImputer(strategy="median")

altman_train = imputer.fit_transform(X_train[altman_cols])
altman_test = imputer.transform(X_test[altman_cols])

Z_train = (
    1.2 * altman_train[:, 0] +
    1.4 * altman_train[:, 1] +
    3.3 * altman_train[:, 2] +
    0.6 * altman_train[:, 3] +
    1.0 * altman_train[:, 4]
)

Z_test = (
    1.2 * altman_test[:, 0] +
    1.4 * altman_test[:, 1] +
    3.3 * altman_test[:, 2] +
    0.6 * altman_test[:, 3] +
    1.0 * altman_test[:, 4]
)

roc_altman = roc_auc_score(y_test, -Z_test)
pr_altman = average_precision_score(y_test, -Z_test)

print("Altman ROC-AUC:", round(roc_altman, 4))
print("Altman PR-AUC:", round(pr_altman, 4))

Altman ROC-AUC: 0.7118
Altman PR-AUC: 0.1572


### Data-Driven Linear Score (Re-estimated Altman)

Rather than using fixed historical weights, we re-estimate coefficients via logistic regression on the same five financial ratios.

In [5]:
log_altman = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

log_altman.fit(altman_train, y_train)

altman_probs = log_altman.predict_proba(altman_test)[:, 1]

roc_log_alt = roc_auc_score(y_test, altman_probs)
pr_log_alt = average_precision_score(y_test, altman_probs)

print("Logistic (5-ratio) ROC-AUC:", round(roc_log_alt, 4))
print("Logistic (5-ratio) PR-AUC:", round(pr_log_alt, 4))

Logistic (5-ratio) ROC-AUC: 0.7341
Logistic (5-ratio) PR-AUC: 0.1441


In [6]:
coefficients = pd.DataFrame({
    "Feature": altman_cols,
    "Logistic_Coefficient": log_altman.coef_[0]
})

coefficients

,Feature,Logistic_Coefficient
0,Attr3,-1.060086
1,Attr6,-0.647235
2,Attr7,-0.941770
3,Attr8,0.005660
4,Attr9,-0.046694


### 📌 Observations & Classical Benchmark Insights

#### 1️⃣ Classical Financial Ratios Retain Signal

Altman-style ratios achieve:

- ROC-AUC ≈ 0.71
- PR-AUC ≈ 0.15

This confirms:

Financial distress theory remains relevant.

---

#### 2️⃣ Data-Driven Re-Weighting Improves Performance

Using logistic regression on the same 5 ratios:

- ROC-AUC improved (~0.73+)

This shows:

Fixed historical coefficients are suboptimal for modern data.

---

#### 3️⃣ Nonlinear Models Still Outperform

XGBoost significantly exceeds classical performance:

- ROC-AUC ~0.97

Indicating:

Bankruptcy risk involves nonlinear and interaction effects beyond linear combination of core ratios.

---

#### 4️⃣ Structural Interpretation

Persistent important ratios include:

- Multi-year gross profitability
- Interest coverage
- Cash flow relative to liabilities
- Sales growth

These represent:

- Profit sustainability
- Debt servicing capacity
- Operational resilience

---

#### 5️⃣ Structural Takeaway

Classical finance provides a meaningful foundation.

However:

Modern bankruptcy prediction benefits significantly from:

- Data-driven coefficient optimization
- Nonlinear modeling
- Interaction-aware ensembles

This notebook establishes financial theory as a strong baseline, not a final solution.